# 🏏 YOLOv8 Cricket Detection Training
Trains a YOLOv8 model on your Roboflow-exported cricket detection dataset (50 epochs).

**Dataset location expected:** `MyDrive/Downloads/cricket detection/` containing `data.yaml`, `train/`, `valid/`, `test/` (each with `images/` and `labels/` subfolders).

Just run all cells top to bottom.

In [ ]:
# 1. Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')


In [ ]:
# 2. Install Ultralytics YOLOv8
!pip install ultralytics -q

import ultralytics
ultralytics.checks()


In [ ]:
# 3. Locate the dataset ZIP in Drive and extract it
# Edit ZIP_PATH manually below if auto-search doesn't find it.

import os, glob

ZIP_CANDIDATES = [
    "/content/drive/MyDrive/Downloads/cricket detection.zip",
    "/content/drive/MyDrive/cricket detection.zip",
    "/content/drive/My Drive/Downloads/cricket detection.zip",
    "/content/drive/My Drive/cricket detection.zip",
]

ZIP_PATH = None
for c in ZIP_CANDIDATES:
    if os.path.isfile(c):
        ZIP_PATH = c
        break

if ZIP_PATH is None:
    # Fallback: search Drive for any zip with 'cricket' in the name
    matches = glob.glob("/content/drive/MyDrive/**/*cricket*.zip", recursive=True)
    if matches:
        ZIP_PATH = matches[0]

if ZIP_PATH is None:
    raise FileNotFoundError(
        "Could not auto-locate the dataset zip. "
        "Set ZIP_PATH manually to the full path of your cricket detection .zip file in Drive."
    )

print("Zip found at:", ZIP_PATH)


In [ ]:
# 4. Extract the zip to local Colab storage (much faster training I/O than reading directly from Drive)
import zipfile

EXTRACT_DIR = "/content/cricket_detection"
os.makedirs(EXTRACT_DIR, exist_ok=True)

with zipfile.ZipFile(ZIP_PATH, "r") as zf:
    zf.extractall(EXTRACT_DIR)

print("Extracted to:", EXTRACT_DIR)
print(os.listdir(EXTRACT_DIR))


In [ ]:
# 5. Locate data.yaml inside the extracted folder
# Handles both cases: zip contains data.yaml at root, or nested inside one extra folder.

def find_data_yaml(root):
    for r, dirs, files in os.walk(root):
        if "data.yaml" in files:
            return r
    return None

DATASET_DIR = find_data_yaml(EXTRACT_DIR)

if DATASET_DIR is None:
    raise FileNotFoundError(
        f"Could not find data.yaml anywhere under {EXTRACT_DIR}. "
        "Check the zip contents with !unzip -l on ZIP_PATH."
    )

print("Dataset found at:", DATASET_DIR)
print(os.listdir(DATASET_DIR))


In [ ]:
# 6. Inspect and fix data.yaml (Roboflow exports often have relative/absolute path issues)
import yaml

yaml_path = os.path.join(DATASET_DIR, "data.yaml")
with open(yaml_path, "r") as f:
    data_cfg = yaml.safe_load(f)

print("Original data.yaml:")
print(data_cfg)

# Rewrite paths to be absolute and point at the images subfolders
data_cfg["path"] = DATASET_DIR
data_cfg["train"] = os.path.join(DATASET_DIR, "train", "images")
data_cfg["val"] = os.path.join(DATASET_DIR, "valid", "images")
if os.path.isdir(os.path.join(DATASET_DIR, "test", "images")):
    data_cfg["test"] = os.path.join(DATASET_DIR, "test", "images")

fixed_yaml_path = "/content/data_fixed.yaml"
with open(fixed_yaml_path, "w") as f:
    yaml.safe_dump(data_cfg, f, sort_keys=False)

print("\nFixed data.yaml written to:", fixed_yaml_path)
print(data_cfg)


In [ ]:
# 7. Sanity check: confirm images/labels exist and counts match
for split in ["train", "valid", "test"]:
    img_dir = os.path.join(DATASET_DIR, split, "images")
    lbl_dir = os.path.join(DATASET_DIR, split, "labels")
    if os.path.isdir(img_dir):
        n_img = len([f for f in os.listdir(img_dir) if f.lower().endswith((".jpg",".jpeg",".png"))])
        n_lbl = len([f for f in os.listdir(lbl_dir) if f.lower().endswith(".txt")]) if os.path.isdir(lbl_dir) else 0
        print(f"{split:6s} -> images: {n_img:5d}   labels: {n_lbl:5d}")
    else:
        print(f"{split:6s} -> not found, skipping")


In [ ]:
# 8. Train YOLOv8 for 50 epochs
from ultralytics import YOLO

# yolov8n = nano (fastest, good for Colab free tier). Swap for yolov8s.pt / yolov8m.pt for higher accuracy.
model = YOLO("yolov8n.pt")

results = model.train(
    data=fixed_yaml_path,
    epochs=50,
    imgsz=640,
    batch=16,
    patience=20,
    project="/content/drive/MyDrive/cricket_detection_runs",
    name="yolov8_cricket",
    exist_ok=True,
)


In [ ]:
# 9. Validate the trained model on the validation set
metrics = model.val()
print(metrics)


In [ ]:
# 10. Run inference on a few test images and preview results
import glob
from IPython.display import Image, display

test_img_dir = os.path.join(DATASET_DIR, "test", "images")
test_images = glob.glob(os.path.join(test_img_dir, "*"))[:5]

if test_images:
    pred = model.predict(
        source=test_images,
        imgsz=640,
        conf=0.25,
        save=True,
        project="/content/drive/MyDrive/cricket_detection_runs",
        name="predict_samples",
        exist_ok=True,
    )
    for r in pred:
        display(Image(filename=r.save_dir + "/" + os.path.basename(r.path)))
else:
    print("No test images found to preview.")


In [ ]:
# 11. Locate best.pt weights (auto-saved to Drive already via 'project' path above)
best_pt = "/content/drive/MyDrive/cricket_detection_runs/yolov8_cricket/weights/best.pt"
print("Best weights saved at:", best_pt)
print("Exists:", os.path.isfile(best_pt))


### Done ✅
- Trained weights: `MyDrive/cricket_detection_runs/yolov8_cricket/weights/best.pt`
- All training curves, confusion matrix, and PR curves are saved in the same run folder.
- Sample predictions are in `MyDrive/cricket_detection_runs/predict_samples/`

To resume/fine-tune later or run inference elsewhere, just load `best.pt` with `YOLO('best.pt')`.